# Support Vector Machines (SVM)

This notebook applies an SVM classifier to the Wine dataset and studies the effect of feature scaling and hyperparameter selection.

Goals:
- Compare an unscaled SVM with a scaled pipeline.
- Tune the regularization parameter and kernel settings.
- Use cross-validation for model selection.
- Evaluate once on a held-out test set.

This is a supplementary, AI-assisted practice exercise. Run and adapt the experiments before using the material as evidence of your own understanding.

In [ ]:
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 1. Prepare the dataset

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 2. Compare unscaled and scaled SVM

SVMs are sensitive to feature scales. The comparison below is illustrative; the scaled pipeline is the recommended approach for this dataset.

In [ ]:
unscaled_model = SVC(kernel="rbf", C=1.0)
unscaled_model.fit(X_train, y_train)
unscaled_accuracy = accuracy_score(y_test, unscaled_model.predict(X_test))

scaled_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1.0))
])
scaled_model.fit(X_train, y_train)
scaled_accuracy = accuracy_score(y_test, scaled_model.predict(X_test))

print(f"Unscaled test accuracy: {unscaled_accuracy:.3f}")
print(f"Scaled test accuracy:   {scaled_accuracy:.3f}")

## 3. Tune the SVM with cross-validation

The scaler is fitted separately inside each cross-validation training fold because it is part of the pipeline.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

param_grid = [
    {
        "svc__kernel": ["linear"],
        "svc__C": [0.1, 1, 10, 100],
    },
    {
        "svc__kernel": ["rbf"],
        "svc__C": [0.1, 1, 10, 100],
        "svc__gamma": ["scale", 0.01, 0.1, 1],
    },
]

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print(f"Best cross-validation accuracy: {search.best_score_:.3f}")

## 4. Final evaluation

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=wine.target_names,
    cmap="Purples"
)
plt.title("SVM — Confusion Matrix")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 5. Further experiments

- Compare linear, RBF, and polynomial kernels.
- Expand or narrow the search ranges for `C` and `gamma`.
- Compare accuracy with macro F1 when class balance is a concern.
- Keep the test set untouched until model selection is complete.